<a href="https://colab.research.google.com/github/bramefendi15-ai/machine-learning-zoomcamp-homework/blob/main/Homework6_Decission_tree_and_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# homework 6- decision tree and ensemble

data set preparation

In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-11-02 22:33:48--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.04s   

2026-11-02 22:33:48 (15.1 MB/s) - ‘car_fuel_efficiency_2026.csv’ saved [635180/635180]



import library

In [4]:

import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
#

save dataset into varibel df

In [5]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(url)

In [6]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [7]:
df.shape

(10000, 11)

In [8]:
df.isnull().sum()

,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0
engine_displacement,0
num_cylinders,0
horsepower,877
vehicle_weight,0
acceleration,264


In [9]:
df = df.fillna(0)

In [10]:
df.isnull().sum()

,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0
engine_displacement,0
num_cylinders,0
horsepower,0
vehicle_weight,0
acceleration,0


## data split and train

In [11]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

change dataframe into matrix

In [12]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=True)

train_dicts = df_train.drop(
    columns=['fuel_efficiency_mpg']
).to_dict(orient='records')

val_dicts = df_val.drop(
    columns=['fuel_efficiency_mpg']
).to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

#Q1- Decision tree regressor

In [13]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=1)

dt.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=1)

In [14]:
feature_idx = dt.tree_.feature[0]

feature_name = dv.get_feature_names_out()[feature_idx]

original_feature = feature_name.split('=')[0]

print(original_feature)               # model_year

model_year


# Q2- Random forest

In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(rmse)                                       # RMSE on validation data set = 1.8370

1.837054054730018


#Q3 - Experiment n_estimator

In [16]:
for n in [10, 50, 100, 150]:

    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(n, round(rmse, 3))                                # 100 has the best RMSE at 1.722

10 1.837
50 1.772
100 1.768
150 1.769


# Q4- finding the best max depth

In [17]:
depths = [10, 15, 20, 25]
n_estimators = [10, 50, 100, 150]

for depth in depths:

    scores = []

    for n in n_estimators:

        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        scores.append(rmse)

    mean_rmse = np.mean(scores)

    print(depth, round(mean_rmse, 3))                        # best max depth shown by 10 with RMSE at 1.755

10 1.755
15 1.784
20 1.788
25 1.786


#Q5- Finding the most important feature

In [18]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, n_estimators=10, n_jobs=-1, random_state=1)

In [19]:
import pandas as pd

feature_names = dv.get_feature_names_out()

importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
})

importance.sort_values(
    by='importance',
    ascending=False
)                                        # model_year showwn to be the most important feature, BUT at the queationary option model_year didn't counted so we choose the other important feature
                                         # which is vehicle_weight

,feature,importance
9,model_year,0.296001
15,vehicle_weight,0.203879
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201
14,origin=USA,0.042144
2,drivetrain=Front-wheel drive,0.020145
11,num_doors,0.019415
5,fuel_type=Diesel,0.017038


#Q6- XBOOST

install XBOOST

In [20]:
!pip install xgboost

In [21]:
import xgboost as xgb

making matrix

In [22]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

make watchlist

In [23]:
watchlist = [
    (dtrain, 'train'),
    (dval, 'validation')
]

In [24]:
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1
}

Ezperiment with ETA 0.3 and 0.1

In [25]:
for eta in [0.3, 0.1]:

    params = xgb_params.copy()
    params['eta'] = eta

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100,
        evals=watchlist
    )

    y_pred = model.predict(dval)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(f"eta: {eta}, RMSE: {rmse:.3f}")

[0]	train-rmse:2.40653	validation-rmse:2.46782
[1]	train-rmse:2.07098	validation-rmse:2.15784
[2]	train-rmse:1.86454	validation-rmse:1.97398
[3]	train-rmse:1.73360	validation-rmse:1.86662
[4]	train-rmse:1.65059	validation-rmse:1.80312
[5]	train-rmse:1.59328	validation-rmse:1.76216
[6]	train-rmse:1.55435	validation-rmse:1.74392
[7]	train-rmse:1.52677	validation-rmse:1.73976
[8]	train-rmse:1.50265	validation-rmse:1.73478
[9]	train-rmse:1.48761	validation-rmse:1.73282
[10]	train-rmse:1.47125	validation-rmse:1.73205
[11]	train-rmse:1.45902	validation-rmse:1.73227
[12]	train-rmse:1.44829	validation-rmse:1.73503
[13]	train-rmse:1.43604	validation-rmse:1.73466
[14]	train-rmse:1.42972	validation-rmse:1.73448
[15]	train-rmse:1.41908	validation-rmse:1.73512
[16]	train-rmse:1.41206	validation-rmse:1.73297
[17]	train-rmse:1.40940	validation-rmse:1.73517
[18]	train-rmse:1.40278	validation-rmse:1.73584
[19]	train-rmse:1.39637	validation-rmse:1.73664
[20]	train-rmse:1.39368	validation-rmse:1.73766
[2